In [ ]:
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("aps/super_glue", "multirc")


In [ ]:
train_ds = ds['train']
val_ds = ds['validation']
test_ds = ds['test']

print(f"Train:      {len(train_ds)} samples")
print(f"Validation: {len(val_ds)} samples")
print(f"Test:       {len(test_ds)} samples")
print(f"\Columns: {train_ds.column_names}")
print(f"\nTrain example:\n{train_ds[0]}")

In [ ]:
from transformers import RobertaTokenizer

tokenizer = RobertaTokenizer.from_pretrained('roberta-base')

def tokenize_function(example):
    return tokenizer(
        example['paragraph'],
        example['question'] + ' ' + example['answer'],
        truncation=True,
    )

cols_to_remove = ['paragraph', 'question', 'answer', 'idx']

tokenized_train = train_ds.map(tokenize_function, remove_columns=cols_to_remove)
tokenized_val   = val_ds.map(tokenize_function,   remove_columns=cols_to_remove)
tokenized_test  = test_ds.map(tokenize_function,  remove_columns=cols_to_remove)

tokenized_train = tokenized_train.rename_column('label', 'labels')
tokenized_val   = tokenized_val.rename_column('label', 'labels')

tokenized_train.set_format('torch')
tokenized_val.set_format('torch')
tokenized_test.set_format('torch')

print("Tokenization concluded")
print(tokenized_train)

In [ ]:
from transformers import RobertaForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
import evaluate
import numpy as np


model = RobertaForSequenceClassification.from_pretrained('roberta-base', num_labels=2)
metric = evaluate.load('accuracy')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

training_args = TrainingArguments(
    output_dir='./checkpoints',
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    logging_steps=100,
)

data_collator = DataCollatorWithPadding(tokenizer)

trainer = Trainer(
    model,
    training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
# Evaluation
train_metrics = trainer.evaluate(tokenized_train, metric_key_prefix='train')
val_metrics   = trainer.evaluate(tokenized_val,   metric_key_prefix='eval')

print(f"Train accuracy: {train_metrics['train_accuracy']:.4f}")
print(f"Val   accuracy: {val_metrics['eval_accuracy']:.4f}")


trainer.save_model('./models/roberta-multirc')
tokenizer.save_pretrained('./models/roberta-multirc')
print("Saved in ./roberta-multirc")

In [ ]:
#TODO: criar a função de teste e avaliar o modelo no test set